In [ ]:
# apktool.bat -version
# apktool.bat d D:\com.jvstudios.gpstracker-254.apk

用apktool将指定目录里的apk解压到apk目录下: 见dicompile_apks.ipynb

In [1]:
# !D:/softwall_install/apktool/apktool.bat --version 2.12.1
# D:/softwall_install/apktool/apktool.bat d D:\TTU\research\location_privacy_compliance\project\open_apk\raw_apks\com.jvstudios.gpstracker-258.apk -f -o D:\TTU
# -s 表示 skip sources，不反编译代码，速度提升 10 倍以上

可直接用在“7zip 解压后的目录”**上的 Python 脚本，做你要的整套流程：

全局粗扫：对解压目录里所有文件（文本/二进制都算）抽取 http(s)://... URL，并记录命中来源文件

按来源分类：dex / res / assets / lib / manifest / 其它

再定位使用点（类/方法）：

优先：如果你有 apktool 解出来的 smali* 目录，它会在 smali 里精确定位到 .method

否则：如果你本机装了 jadx（推荐 jadx-cli），脚本会自动调用它，解析反编译后的代码，定位到类/方法附近

输出：同时输出 JSON（最方便读）+ CSV（方便 Excel/统计）

我建议你两者都输出：JSON 用来“看结构和证据链”、CSV 用来“筛选/排序/统计/画表”。

In [5]:
import argparse
import csv
import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
from collections import defaultdict
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Tuple, Optional
import re, json, csv, hashlib
from collections import defaultdict
from datetime import datetime
from urllib.parse import urlparse
import re
import pandas as pd
from collections import Counter

In [6]:
# URL_BYTES_RE = re.compile(
#     br"https?://[A-Za-z0-9\-\._~:/\?#\[\]@!\$&'\(\)\*\+,;=%]+"
# )

# 先宽松抓取，但不跨引号、空白、尖括号、反斜杠
URL_BYTES_RE = re.compile(
    rb'https?://[^\s"\'<>{}\\|`]+',
    re.IGNORECASE
)

# 用于把一个粘连串里再次拆成多个 http/https 起点
HTTP_SPLIT_RE = re.compile(r'https?://', re.IGNORECASE)

# 常见“网页/资源 URL”后缀，可按需加
# LIKELY_URL_END_RE = re.compile(
#     r'''(?ix)
#     ^
#     (
#         https?://.*?
#         (?:
#             \.html? |
#             \.xhtml |
#             \.php |
#             \.asp(?:x)? |
#             \.jsp |
#             \.json |
#             \.xml |
#             \.js |
#             \.css |
#             \.svg |
#             \.png |
#             \.jpg |
#             \.jpeg |
#             \.webp |
#             \.gif |
#             \.ico |
#             / |
#             \? |
#             \#
#         )
#     )
#     '''
# )
LIKELY_URL_END_RE = re.compile(
    r'''(?ix)
    ^
    (
        https?://.*?
        (?:
            \.html? |
            ...
            / |
            \? |
            \#
        )
    )
    '''
)

# ---- smali precise mapping ----
SMALI_CLASS_RE = re.compile(r"^\.class\b.*\s(L.+;)\s*$")
SMALI_METHOD_RE = re.compile(r"^\.method\b(.*)$")
SMALI_END_METHOD_RE = re.compile(r"^\.end method\b")
SMALI_CONST_STRING_RE = re.compile(r'^\s*const-string(?:/jumbo)?\s+[^,]+,\s+"(.*)"\s*$')

# ---- 1) 规则库：第三方域名 & 包名关键字（可按你项目继续加） ----
THIRD_PARTY_DOMAIN_HINTS = [
    "google.com", "googleapis.com", "gstatic.com",
    "firebase", "firebasestorage.googleapis.com",
    "mapbox.com", "api.mapbox.com",
    "doubleclick.net", "googlesyndication.com",
    "facebook.com", "fbcdn.net",
    "appsflyer.com", "adjust.com",
    "branch.io", "segment.com", "amplitude.com", "mixpanel.com",
    "crashlytics", "sentry.io",
    "app-measurement.com",
]

# smali/jadx 文件路径里出现这些，多半是第三方 SDK 代码
THIRD_PARTY_CODE_HINTS = [
    "/com/google/",
    "/com/firebase/",
    "/com/mapbox/",
    "/com/facebook/",
    "/com/appsflyer/",
    "/com/adjust/",
    "/io/sentry/",
    "/com/segment/",
    "/com/amplitude/",
    "/com/mixpanel/",
    "/okhttp3/", "/retrofit2/",
    "/com/airbnb/",  # 视情况
]

# ---- 3) 非“给用户看的网页”类型（常见资源后缀）----
NON_PAGE_EXT = (".js",".css",".png",".jpg",".jpeg",".webp",".gif",".svg",".ico",
                ".json",".mp3",".mp4",".wav",".zip",".apk",".dex",".so")

# ---- 2) API / 埋点 endpoint patterns ----
API_PATH_PATTERNS = [
    r"/api(/|$)", r"/v\d+(/|$)", r"/graphql(/|$)",
    r"/track(/|$)", r"/collect(/|$)", r"/event(s)?(/|$)",
    r"/log(s)?(/|$)", r"/analytics(/|$)", r"/metrics(/|$)",
    r"/measurement(/|$)", r"/telemetry(/|$)"
]
API_PATH_RE = re.compile("|".join(API_PATH_PATTERNS), re.IGNORECASE)

In [7]:
# URL 扫描 + 分类 + smali 精确定位（class/method/line）
def normalize_url(u: str) -> str:
    return u.rstrip('"\')]>},.;:')

def is_probably_valid_url(u: str) -> bool:
    try:
        p = urlparse(u)
        return p.scheme in ("http", "https") and bool(p.netloc)
    except Exception:
        return False

def trim_concatenated_tail(u: str) -> str:
    """
    优先截掉后续拼接的第二个 http/https，
    再处理尾部无关字符。
    """
    u = normalize_url(u)

    # 1) 先找后面是否还拼了一个新的 http/https
    m2 = re.search(r'(?i)https?://', u[8:])   # 跳过开头自身
    if m2:
        u = u[:8 + m2.start()]

    u = normalize_url(u)

    # 2) 如果尾部像驼峰函数名/变量名，再尝试按资源后缀裁掉
    m = re.match(
        r'(?i)^(.+?\.(?:html?|xhtml|php|aspx?|jsp|json|xml|js|css|svg|png|jpg|jpeg|webp|gif|ico))(?:[A-Z_].*)?$',
        u
    )
    if m:
        candidate = m.group(1)
        if is_probably_valid_url(candidate):
            return candidate

    return u

def classify_source(rel_path: str) -> str:
    p = rel_path.replace("\\", "/")
    name = p.lower()
    if name.endswith(".dex"):
        return "dex"
    if name.endswith(".so"):
        return "lib_native"
    if p.startswith("assets/"):
        return "assets"
    if p.startswith("res/"):
        return "res"
    if name.endswith("androidmanifest.xml"):
        return "manifest"
    if name.endswith(".arsc"):
        return "resources_arsc"
    if p.startswith("smali") and name.endswith(".smali"):
        return "smali"
    if p.startswith("unknown/"):
        return "unknown"
    return "other"

def file_sha1_1mb(path: Path) -> str:
    h = hashlib.sha1()
    with path.open("rb") as f:
        h.update(f.read(1024 * 1024))
    return h.hexdigest()

def looks_like_real_page_or_asset_1(u: str) -> bool:
    try:
        p = urlparse(u)
        path = (p.path or "").lower()
        return (
            "privacy" in u.lower()
            or path.endswith((".html", ".htm", ".php", ".aspx", ".jsp", ".json", ".xml",
                              ".png", ".jpg", ".jpeg", ".svg", ".webp", ".css", ".js"))
            or "/" in path
        )
    except Exception:
        return False
    
def looks_like_real_page_or_asset(u: str) -> bool:
    try:
        p = urlparse(u)
        host = (p.netloc or "").lower()
        path = (p.path or "").lower()

        if not host:
            return False

        return (
            "privacy" in u.lower()
            or path.endswith((
                ".html", ".htm", ".php", ".aspx", ".jsp", ".json", ".xml",
                ".png", ".jpg", ".jpeg", ".svg", ".webp", ".css", ".js"
            ))
            or len(path) > 1
        )
    except Exception:
        return False

HTTP_START_RE = re.compile(rb'https?://', re.IGNORECASE)

def extract_urls_from_bytes(data: bytes): #, max_window: int = 2048
    for m in URL_BYTES_RE.finditer(data):
        # start = m.start()
        # chunk = data[start:start + max_window]
        # try:
        #     raw = chunk.decode("utf-8", errors="ignore")
        # except Exception:
        #     raw = chunk.decode("latin1", errors="ignore")
        # # 碰到明显分隔符就截断
        # raw = re.split(r'[\s"\'<>{}\\|`]', raw, maxsplit=1)[0]
        # raw = trim_concatenated_tail(raw)
        # if not raw:
        #     continue
        # if not is_probably_valid_url(raw):
        #     continue
        # if not looks_like_real_page_or_asset(raw):
        #     continue

        # yield raw, start


        try:
            raw = m.group(0).decode("utf-8", errors="ignore")
        except Exception:
            raw = m.group(0).decode("latin1", errors="ignore")

        raw = normalize_url(raw)
        if not raw:
            continue

        # 一个命中里如果又出现了多个 http/https，拆开
        starts = [x.start() for x in HTTP_SPLIT_RE.finditer(raw)]
        if not starts:
            continue

        

        starts.append(len(raw))

        for i in range(len(starts) - 1):
            piece = raw[starts[i]:starts[i + 1]]

            # 保险起见，只保留从 http(s):// 开始的位置
            m2 = HTTP_SPLIT_RE.search(piece)
            if not m2:
                continue
            piece = piece[m2.start():]

            piece = trim_concatenated_tail(piece)

            if not is_probably_valid_url(piece):
                continue

            if not looks_like_real_page_or_asset(piece):
                continue

            if not piece:
                continue
            

            yield piece, m.start() + starts[i]

def scan_all_files(root: Path, max_mb: int = 50):
    max_bytes = max_mb * 1024 * 1024
    recs = []
    for p in root.rglob("*"):
        if not p.is_file():
            continue
        try:
            size = p.stat().st_size
            if size > max_bytes:
                continue
            data = p.read_bytes()
        except Exception:
            continue

        hits = list(extract_urls_from_bytes(data))
        if not hits:
            continue

        rel = str(p.relative_to(root)).replace("\\", "/")
        src = classify_source(rel)
        sha = file_sha1_1mb(p)

        for url, off in hits:
            recs.append({
                "url": url,
                "file": rel,
                "source": src,
                "offset": off,
                "size_bytes": size,
                "file_sha1_1mb": sha,
            })
    # dedup by (url, file, offset)
    seen = set()
    out = []
    for r in recs:
        k = (r["url"], r["file"], r["offset"])
        if k in seen:
            continue
        seen.add(k)
        out.append(r)
    return out

def map_urls_in_smali(root: Path, target_urls: set):
    usage = defaultdict(list)
    smali_dirs = [p for p in root.iterdir() if p.is_dir() and p.name.startswith("smali")]
    for sd in smali_dirs:
        for smali_file in sd.rglob("*.smali"):
            try:
                lines = smali_file.read_text(encoding="utf-8", errors="ignore").splitlines()
            except Exception:
                continue

            cur_class = None
            cur_method = None
            in_method = False

            for idx, line in enumerate(lines, start=1):
                m = SMALI_CLASS_RE.match(line)
                if m:
                    cur_class = m.group(1)

                m = SMALI_METHOD_RE.match(line)
                if m:
                    cur_method = m.group(1).strip()
                    in_method = True

                if SMALI_END_METHOD_RE.match(line):
                    in_method = False
                    cur_method = None

                m = SMALI_CONST_STRING_RE.match(line)
                if m:
                    s = m.group(1)
                    if "http://" in s or "https://" in s:
                        for url, _ in extract_urls_from_bytes(s.encode("utf-8", errors="ignore")):
                            if url in target_urls:
                                usage[url].append({
                                    "engine": "smali",
                                    "smali_file": str(smali_file.relative_to(root)).replace("\\", "/"),
                                    "class": cur_class,
                                    "method": cur_method if in_method else None,
                                    "line": idx,
                                    "snippet": line.strip(),
                                })
    return usage

# 快速筛“疑似隐私政策 URL”
def urls_contain_keyword(urls_sorted):
    privacy_keywords = ("privacy", "privacy-policy", "privacypolicy")

    candidates = []
    for u in urls_sorted:
        lu = u.lower()
        if any(k in lu for k in privacy_keywords):
            candidates.append(u)

    print("Privacy-like candidates:", len(candidates))
    for u in candidates[:50]:
        print("-", u)
    return candidates


# ---- 2) API / 埋点 endpoint patterns ----

# ---- 3) 非“给用户看的网页”类型（常见资源后缀）----

def host_of(url: str) -> str:
    try:
        return (urlparse(url).netloc or "").lower()
    except Exception:
        return ""

def path_of(url: str) -> str:
    try:
        return (urlparse(url).path or "").lower()
    except Exception:
        return ""

def looks_like_api_or_tracking(url: str) -> bool:
    p = path_of(url)
    if API_PATH_RE.search(p):
        return True
    # query里常见埋点参数也算（可选）
    q = (urlparse(url).query or "").lower()
    if any(k in q for k in ["utm_", "gclid", "fbclid", "event", "click", "adid"]):
        return True
    return False
    

def looks_like_third_party_domain(url: str) -> bool:
    h = host_of(url)
    return any(h.endswith(d) or d in h for d in THIRD_PARTY_DOMAIN_HINTS)

def location_looks_like_third_ad(url: str, file_path) -> bool:
    # 1. 定义第三方/广告的关键词黑名单
    AD_KEYWORDS = ['ad-', 'ads', 'tracker', 'analytics', 'doubleclick', 'google-ad']
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    for item in data.get('urls', []):
        if item.get('url') == url:
            # 找到匹配后，遍历该 url 下的所有 hits
            for hit in item.get('hits', []):
                file_path = hit.get('file').lower()
                source_type = hit.get('source').lower()

                # 2. 判断是否包含黑名单关键词
                is_third_party = any(kw in file_path.lower() for kw in AD_KEYWORDS)
                if is_third_party:
                    return True
    return False            


def has_non_page_extension(url: str) -> bool:
    p = path_of(url)
    return any(p.endswith(ext) for ext in NON_PAGE_EXT)

def usage_is_third_party(usage_map,url: str) -> bool:
    """
    如果该 URL 的 usage（smali 文件路径）明显落在第三方包路径里，就判为第三方。
    注意：这比域名更可靠，因为有些 app 的 policy 也可能托管在第三方域名上。
    """
    for u in usage_map.get(url, []):
        f = (u.get("smali_file") or "").lower()
        if any(h in f for h in THIRD_PARTY_CODE_HINTS):
            return True
    return False

def app_package_hint_from_root(root: Path) -> str:
    """
    粗略推断 app 的包名：从 smali* 下找出现频率最高的 top-level package 之一
    （也可以直接从 AndroidManifest.xml 解析 package=，这里先不给你搞复杂）
    """
    # 如果你想更准：解析 out_jv/AndroidManifest.xml 里的 package=""
    return ""

# APP_PKG_HINT = app_package_hint_from_root(ROOT)

def usage_is_app_code(usage_map, app_prefixes, url: str) -> bool:
    """
    如果 usage 的 smali_file 看起来在 app 自己包名路径下（如 /com/jvstudios/），则更像 app 自己使用。
    这里你可以手动改成你的 app 包前缀，比如 "/com/jvstudios/"。
    """
    # 手动写包名前缀最靠谱（示例：jvstudios）
    # app_prefixes = ["com/locator/gpstracker/phone"]  # <- 你可以根据不同 app 改成 /com/locator/ 等 "/com/jvstudios/gpstracker/"
    for u in usage_map.get(url, []):
        f = (u.get("smali_file") or "").lower()
        if any(pref in f for pref in app_prefixes):
            return True
    return False

In [38]:
# path = "two_examples/dicompile"
# folders = [item for item in os.listdir(path) if os.path.isdir(os.path.join(path, item))]
# folders

# new_folders = [[f"/{f.split('-')[0].replace('.', '/')}/"] for f in folders]
# new_folders

# # [['/com/jvstudios/gpstracker/'],
# #  ['/com/jvstudios/gpstracker/'],
# #  ['/com/locator/gpstracker/phone/'],
# #  ['/com/locator/gpstracker/phone/']]

In [ ]:
### AA1_first_batch
# [
# "1122apk\dicompile1122apk\AA1_first_batch\AA1_first_328_batch"
# 1122apk\dicompile1122apk\AA1_first_batch\AA2_second_100_batch
# 1122apk\dicompile1122apk\AA1_first_batch\AA3_third_100_batch
# 1122apk\dicompile1122apk\AA1_first_batch\AA4_forth_100_batch
# 1122apk\dicompile1122apk\AA1_first_batch\AA5_fifth_100_batch
# 1122apk\dicompile1122apk\AA1_first_batch\AA6_sisth_74_batch
# ]

# [
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA1_first_328_batch
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA2_second_100_batch
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA3_third_100_batch
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA4_forth_100_batch
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA5_fifth_100_batch
# 1122apk\1122apk_privacy_policy_url\AA1_first_batch\AA6_sisth_100_batch
# ]

### AA2_second_batch
# 1122apk\dicompile1122apk\AA2_second_batch\AA1_first_100_batch
# 1122apk\dicompile1122apk\AA2_second_batch\AA2_second_100_batch
# 1122apk\dicompile1122apk\AA2_second_batch\AA3_third_100_batch
# 1122apk\dicompile1122apk\AA2_second_batch\AA4_forth_100_batch
# 1122apk\dicompile1122apk\AA2_second_batch\AA5_fifth_100_batch
# 1122apk\dicompile1122apk\AA2_second_batch\AA6_sisth_100_batch
# 1122apk\dicompile1122apk\AA2_second_batch\AA7_seventh_100_batch

# 1122apk\1122apk_privacy_policy_url\AA2_second_batch\AA1_first_100_batch
# 1122apk\1122apk_privacy_policy_url\AA2_second_batch\AA2_second_100_batch
# 1122apk\1122apk_privacy_policy_url\AA2_second_batch\AA3_third_100_batch
# 1122apk\1122apk_privacy_policy_url\AA2_second_batch\AA4_forth_100_batch
# 1122apk\1122apk_privacy_policy_url\AA2_second_batch\AA5_fifth_100_batch
# 1122apk\1122apk_privacy_policy_url\AA2_second_batch\AA6_sisth_100_batch
# 1122apk\1122apk_privacy_policy_url\AA2_second_batch\AA7_seventh_100_batch

### AA3_third_time
# 1122apk\dicompile1122apk\AA3_third_batch\AA1_first_100_batch
# 1122apk\dicompile1122apk\AA3_third_batch\AA2_second_100_batch
# 1122apk\dicompile1122apk\AA3_third_batch\AA3_third_100_batch
# 1122apk\dicompile1122apk\AA3_third_batch\AA4_forth_100_batch
# 1122apk\dicompile1122apk\AA3_third_batch\AA5_fifth_100_batch
# 1122apk\dicompile1122apk\AA3_third_batch\AA6_sisth_100_batch
# 1122apk\dicompile1122apk\AA3_third_batch\AA7_seventh_100_batch

# 1122apk\1122apk_privacy_policy_url\AA3_third_batch\AA1_first_100_batch
# 1122apk\1122apk_privacy_policy_url\AA3_third_batch\AA2_second_100_batch
# 1122apk\1122apk_privacy_policy_url\AA3_third_batch\AA3_third_100_batch
# 1122apk\1122apk_privacy_policy_url\AA3_third_batch\AA4_forth_100_batch
# 1122apk\1122apk_privacy_policy_url\AA3_third_batch\AA5_fifth_100_batch
# 1122apk\1122apk_privacy_policy_url\AA3_third_batch\AA6_sisth_100_batch
# 1122apk\1122apk_privacy_policy_url\AA3_third_batch\AA7_seventh_100_batch


### AA4_forth_time
# 1122apk\dicompile1122apk\AA4_fourth_batch

# 1122apk\1122apk_privacy_policy_url\AA4_fourth_batch

In [8]:
path = "1122apk\dicompile1122apk\AA4_fourth_batch"  # "1122apk/20_examples/test1apk" # "1122apk/dicompile1122apk" # "two_examples/dicompile" # "1122apk\dicompile1122apk"


folders = [item for item in os.listdir(path) if os.path.isdir(os.path.join(path, item))]
for folder in folders:
    # cur_path = Path(rf"{path}/{folder}")
    # print(cur_path)
    # 改成你的 apktool 输出目录
    ROOT = Path(rf"{path}/{folder}")  # 例如 r"D:\apktool_out\out_jv" "apk/com.jvstudios.gpstracker-258" "apk/com.locator.gpstracker.phone-153-apktool"
    print(ROOT)
    app_name = folder
    prefix_path = folder.split('-')[0].replace('.', '/')
    app_prefixes = [f"/{prefix_path}/"]
    print(app_prefixes)
    OUT_DIR = Path(r"1122apk\1122apk_privacy_policy_url\AA4_fourth_batch") 
    # [] 
    OUT_DIR.mkdir(parents=True, exist_ok=True)



    JSON_PATH = OUT_DIR / f"{app_name}_urls.json"
    CSV_PATH  = OUT_DIR / f"{app_name}_urls.csv"

    Original_Path = OUT_DIR / f"{app_name}_original_urls.csv"

    # print("ROOT:", ROOT.resolve())
    # print("OUT :", OUT_DIR.resolve())

    # ===== URL 扫描 + 分类 + smali 精确定位（class/method/line）=====
    records = scan_all_files(ROOT, max_mb=50)
    target_urls = set(r["url"] for r in records)
    usage_map = map_urls_in_smali(ROOT, target_urls)

    print("Total hits:", len(records))
    print("Unique URLs:", len(target_urls))
    print("URLs with smali usage:", sum(1 for u in target_urls if u in usage_map))
    # for url in target_urls:
    #     print(url)
    # JSON_PATH.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")

    # ==== 筛选含有关键词的urls，然后对其按来源分类 + 输出 JSON / CSV ====
    by_source = Counter(r["source"] for r in records)
    print("By source:", by_source)
    urls_sorted = sorted(target_urls)
    candidates = urls_contain_keyword(urls_sorted)
    payload = {
        "generated_at": datetime.now().isoformat(timespec="seconds"),
        "summary": {
            "total_hits": len(records),
            "unique_urls": len(candidates),
            "by_source": dict(by_source),
        },
        "urls": [
            {
                "url": u,
                "hits": [r for r in records if r["url"] == u],
                "usage": usage_map.get(u, []),
            }
            for u in candidates
        ],
    }

    # ===== 输出 JSON / CSV ====
    JSON_PATH.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
    print("Wrote JSON:", JSON_PATH)

    # CSV (flat)
    fields = [
        "url","source","file","offset","size_bytes","file_sha1_1mb",
        "usage_engine","usage_smali_file","usage_class","usage_method","usage_line"
    ]
    with CSV_PATH.open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fields)
        w.writeheader()
        for r in records:
            usages = usage_map.get(r["url"], [])
            if not usages:
                w.writerow({**r,
                            "usage_engine":"",
                            "usage_smali_file":"",
                            "usage_class":"",
                            "usage_method":"",
                            "usage_line":""})
            else:
                for u in usages:
                    w.writerow({**r,
                                "usage_engine": u.get("engine",""),
                                "usage_smali_file": u.get("smali_file",""),
                                "usage_class": u.get("class",""),
                                "usage_method": u.get("method",""),
                                "usage_line": u.get("line","")})

    print("Wrote CSV :", CSV_PATH)



    # ---- 排除并给排除理由 ----
    kept = []
    dropped = []

    for url in candidates:
        reasons_drop = []
        reasons_keep = []

        if has_non_page_extension(url):
            reasons_drop.append("non-page extension (asset/media/script)")

        # API/埋点优先踢
        if looks_like_api_or_tracking(url):
            reasons_drop.append("looks like API/tracking endpoint")

        # 如果 usage 明显在第三方 SDK 包里，踢
        if usage_is_third_party(usage_map, url):
            reasons_drop.append("used inside third-party SDK package (smali path)")

        # 域名像第三方，不直接踢（因为 policy 也可能托管在第三方），但作为弱信号
        if looks_like_third_party_domain(url):
            reasons_drop.append("third-party hosting domain (weak signal)")

        # url所在文件路径看起来像第三方或者广告，踢
        if location_looks_like_third_ad(url, JSON_PATH):
            reasons_drop.append("location of url looks like third or ad")
    

        # 反过来：如果它在 app 自己代码路径下被引用，是强保留信号
        if usage_is_app_code(usage_map, app_prefixes, url):
            reasons_keep.append("referenced from app package code (smali path)")
            # 如果有强保留信号，就把“弱信号”从 drop 里剔除（比如第三方域名）
            reasons_drop = [r for r in reasons_drop if "third-party hosting domain" not in r]

        # 最终判定：有任何“硬踢”理由且没有强保留，就丢弃
        hard_drop = any(r in reasons_drop for r in [
            "non-page extension (asset/media/script)",
            "looks like API/tracking endpoint",
            "used inside third-party SDK package (smali path)",
            "location of url looks like third or ad",
        ])

        if hard_drop and not reasons_keep:
            dropped.append({"url": url, "drop_reasons": reasons_drop})
        else:
            kept.append({"url": url, "keep_reasons": reasons_keep, "warnings": reasons_drop})

    print("Candidates:", len(candidates))
    print("Kept      :", len(kept))
    print("Dropped   :", len(dropped))

    print("\n--- KEPT (top 30) ---")
    for x in kept[:30]:
        print("-", x["url"])
        if x["keep_reasons"]:
            print("   keep:", "; ".join(x["keep_reasons"]))
        if x["warnings"]:
            print("   warn:", "; ".join(x["warnings"]))

    print("\n--- DROPPED (top 30) ---")
    for x in dropped[:30]:
        print("-", x["url"])
        print("   drop:", "; ".join(x["drop_reasons"]))

    df_kept = pd.DataFrame(kept)
    df_dropped = pd.DataFrame(dropped)

    (df_kept).to_csv(OUT_DIR / f"{app_name}_privacy_candidates_kept.csv", index=False, encoding="utf-8")
    (df_dropped).to_csv(OUT_DIR / f"{app_name}_privacy_candidates_dropped.csv", index=False, encoding="utf-8")

    (OUT_DIR / f"{app_name}_privacy_candidates_kept.json").write_text(json.dumps(kept, indent=2, ensure_ascii=False), encoding="utf-8")
    (OUT_DIR / f"{app_name}_privacy_candidates_dropped.json").write_text(json.dumps(dropped, indent=2, ensure_ascii=False), encoding="utf-8")

    print("Wrote:", OUT_DIR / f"{app_name}_privacy_candidates_kept.csv")
    print("Wrote:", OUT_DIR / f"{app_name}_privacy_candidates_dropped.csv")

1122apk\dicompile1122apk\AA4_fourth_batch\thug.life.photo.sticker.maker-588
['/thug/life/photo/sticker/maker/']
Total hits: 1194
Unique URLs: 105
URLs with smali usage: 58
By source: Counter({'res': 1076, 'smali': 79, 'unknown': 28, 'assets': 9, 'manifest': 2})
Privacy-like candidates: 2
- https://firebase.google.com/support/privacy/init-options
- https://sites.google.com/flocmedia.com/thuglifepiceditorprivacypolicy/home
Wrote JSON: 1122apk\1122apk_privacy_policy_url\AA4_fourth_batch\thug.life.photo.sticker.maker-588_urls.json
Wrote CSV : 1122apk\1122apk_privacy_policy_url\AA4_fourth_batch\thug.life.photo.sticker.maker-588_urls.csv
Candidates: 2
Kept      : 2
Dropped   : 0

--- KEPT (top 30) ---
- https://firebase.google.com/support/privacy/init-options
   warn: third-party hosting domain (weak signal)
- https://sites.google.com/flocmedia.com/thuglifepiceditorprivacypolicy/home
   warn: third-party hosting domain (weak signal)

--- DROPPED (top 30) ---
Wrote: 1122apk\1122apk_privacy_po

遍历每一个apkname-versioncode_privacy_candidates_kept.json,把"keep_reasons"为"referenced from app package code (smali path)"的pp url取出来，最后保存为json吧，apkname-versioncode: [pp urls]

In [36]:
# 1. 设定基础路径 OUT_DIR = Path(r"1122apk/1122apk_privacy_policy_url")
base_path = Path(r"two_examples/1122apk_privacy_policy_url") 
output_file = Path(rf"{base_path}/out/extracted_pp_urls.json")
# print(output_file)
# output_file.mkdir(parents=True, exist_ok=True)

results = {}

# 2. 遍历目录下所有符合 *_privacy_candidates_kept.json 后缀的文件
for json_file in base_path.glob("*_privacy_candidates_kept.json"):
    
    # 提取 key (apkname-versioncode)，即去掉 "_privacy_candidates_kept.json" 部分
    app_key = json_file.name.replace("_privacy_candidates_kept.json", "")
    
    pp_urls = []
    
    try:
        with open(json_file, 'r', encoding='utf-8') as f:
            data = json.load(f)
            
            # 假设数据是列表格式，遍历每一个项
            for item in data:
                # 检查 keep_reasons 是否匹配目标字符串
                # 注意：这里使用了 in 或 ==，根据实际 json 格式（可能是列表也可能是字符串）
                reasons = item.get("keep_reasons", [])
                
                # 兼容处理：如果 keep_reasons 是列表则检查是否存在，如果是字符串则直接比较
                target_reason = "referenced from app package code (smali path)"
                if (isinstance(reasons, list) and target_reason in reasons) or (reasons == target_reason):
                    url = item.get("url")
                    if url:
                        pp_urls.append(url)
        
        # 只有找到符合条件的 URL 时才存入结果
        if pp_urls:
            results[app_key] = pp_urls
            
    except Exception as e:
        print(f"处理文件 {json_file.name} 时出错: {e}")

# 3. 将结果保存为新的 JSON 文件

with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=4, ensure_ascii=False)

# print(f"提取完成！结果已保存至: {output_file}")
print(f"共处理了 {len(results)} 个应用的有效数据。")

共处理了 4 个应用的有效数据。


在wayback machine里过去时间段该pp有没有 -- 要补这块代码

遍历上面保存的文档(found_privacy_policy_url.txt)，遍历privacy policy url，在Wayback Machine里面搜索该url的历史记录，看2024年10月份间隔1年有没有记录。

遍历上面保存的文档1122apk/1122apk_privacy_policy_url/extracted_pp_urls.json")，key是apkname，value是对应的privacy policy url，读取里面每一个apk的privacy policy url，在Wayback Machine里面搜索该url的历史记录，看2024年10月份间隔上下1年有没有记录。

遍历 extracted_pp_urls.json
2️⃣ 查询 Wayback
3️⃣ 找到 最接近 2024-10-15 的 snapshot
4️⃣ 把结果写入 CSV（中间结果）
5️⃣ 如果存在 snapshot → 下载 TXT, 而不是HTML

In [8]:
import os
import json
import csv
import time
import random
import requests
from urllib.parse import urlparse
import re
from html import unescape

In [ ]:
INPUT_JSON = r"1122apk/two_examples/1122apk_privacy_policy_url/out/extracted_pp_urls.json" #r"1122apk/1122apk_privacy_policy_url/out/extracted_pp_urls.json"
OUTPUT_DIR = r"1122apk/two_examples/1122apk_privacy_policy_url/pp_txt_2024_10" #r"1122apk/1122apk_privacy_policy_url/pp_html_2024_10"
CSV_FILE = r"1122apk/two_examples/1122apk_privacy_policy_url/out/pp_wayback_result.csv" #"1122apk/1122apk_privacy_policy_url/out/pp_wayback_result.csv"

# OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [14]:
FROM_TS = "20231001"
TO_TS   = "20251031"
TARGET_TS = "20241015000000"   # 用于找“离 2024-10-15 最近”的记录

CDX_API = "https://web.archive.org/cdx/search/cdx"

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0"
})

In [15]:
def normalize_url(url):
    if not url:
        return None
    url = url.strip()
    if not url.startswith(("http://", "https://")):
        url = "http://" + url
    return url


def ts_distance(a, b):
    return abs(int(a) - int(b))


def find_closest_snapshot(url):

    params = {
        "url": url,
        "from": FROM_TS,
        "to": TO_TS,
        "output": "json",
        "fl": "timestamp,original",
        "collapse": "digest",
        "filter": "statuscode:200"
    }

    r = session.get(CDX_API, params=params, timeout=30)

    if r.status_code != 200:
        return None

    data = r.json()

    if len(data) <= 1:
        return None

    header = data[0]
    rows = data[1:]

    records = []

    for row in rows:
        item = dict(zip(header, row))
        records.append(item)

    closest = min(records, key=lambda x: ts_distance(x["timestamp"], TARGET_TS))

    return closest



def html_to_text(html):
    # 去掉 script/style
    html = re.sub(r"(?is)<script.*?>.*?</script>", " ", html)
    html = re.sub(r"(?is)<style.*?>.*?</style>", " ", html)

    # 常见块标签换行，避免全挤一起
    html = re.sub(r"(?i)</p>|<br\s*/?>|</div>|</li>|</tr>|</h[1-6]>", "\n", html)
    html = re.sub(r"(?i)<li[^>]*>", "- ", html)

    # 去所有标签
    text = re.sub(r"(?s)<[^>]+>", " ", html)

    # HTML 实体反转义
    text = unescape(text)

    # 清理多余空白
    text = re.sub(r"\r\n|\r", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n\s*\n\s*\n+", "\n\n", text)

    return text.strip()


def download_snapshot_1(apk_name, snapshot):

    ts = snapshot["timestamp"]
    url = snapshot["original"]

    wayback_url = f"https://web.archive.org/web/{ts}/{url}"

    out_file = os.path.join(OUTPUT_DIR, apk_name + ".html")

    try:
        r = session.get(wayback_url, timeout=60)

        if r.status_code != 200:
            return False

        with open(out_file, "wb") as f:
            f.write(r.content)

        return True

    except Exception:
        return False

def download_snapshot(apk_name, snapshot):

    ts = snapshot["timestamp"]
    url = snapshot["original"]

    wayback_url = f"https://web.archive.org/web/{ts}/{url}"

    out_file = os.path.join(OUTPUT_DIR, apk_name + ".txt")

    try:
        r = session.get(wayback_url, timeout=60)

        if r.status_code != 200:
            return False

        # 尽量按 requests 猜到的编码解码
        r.encoding = r.apparent_encoding or r.encoding
        text = html_to_text(r.text)

        with open(out_file, "w", encoding="utf-8") as f:
            f.write(text)

        return True

    except Exception:
        return False


def append_csv(row):

    file_exists = os.path.exists(CSV_FILE)

    with open(CSV_FILE, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=[
                "apk_name",
                "pp_url",
                "timestamp",
                "wayback_url",
                "has_snapshot",
                "downloaded"
            ]
        )

        if not file_exists:
            writer.writeheader()

        writer.writerow(row)

In [16]:
with open(INPUT_JSON, "r", encoding="utf-8") as f:
    data = json.load(f)

total = len(data)

print("Total:", total)

for i, (apk_name, urls) in enumerate(data.items(), 1):

    if not urls:
        url = None
    else:
        url = urls[0]


    url = normalize_url(url)

    print(f"[{i}/{total}] {apk_name}")

    if not url:

        append_csv({
            "apk_name": apk_name,
            "pp_url": "",
            "timestamp": "",
            "wayback_url": "",
            "has_snapshot": False,
            "downloaded": False
        })

        continue

    snapshot = find_closest_snapshot(url)

    if not snapshot:

        print("   no snapshot")

        append_csv({
            "apk_name": apk_name,
            "pp_url": url,
            "timestamp": "",
            "wayback_url": "",
            "has_snapshot": False,
            "downloaded": False
        })

        continue

    ts = snapshot["timestamp"]
    wayback_url = f"https://web.archive.org/web/{ts}/{url}"

    print("   snapshot:", ts)

    downloaded = download_snapshot(apk_name, snapshot)

    append_csv({
        "apk_name": apk_name,
        "pp_url": url,
        "timestamp": ts,
        "wayback_url": wayback_url,
        "has_snapshot": True,
        "downloaded": downloaded
    })

    time.sleep(1)

print("Done.")

Total: 4
[1/4] com.jvstudios.gpstracker-254
   snapshot: 20241224121427
[2/4] com.jvstudios.gpstracker-258
   snapshot: 20241224121427
[3/4] com.locator.gpstracker.phone-153
   snapshot: 20241224133705
[4/4] com.locator.gpstracker.phone-154
   snapshot: 20241224133705
Done.


不用的代码

In [ ]:
import json
import time
from pathlib import Path
from collections import defaultdict
import requests

import random
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

import re, json, csv, hashlib
from datetime import datetime
from collections import Counter

In [ ]:
ROOT = Path(r"1122apk/test")
FOUND_TXT = Path("1122apk/1122apk_privacy_policy_url/out/found_privacy_policy_url.txt")

OUT_HAS = Path("1122apk/1122apk_privacy_policy_url/out/wayback_has_2024_10.txt")
OUT_NO = Path("1122apk/1122apk_privacy_policy_url/out/wayback_no_2024_10.txt")
OUT_ERR = Path("1122apk/1122apk_privacy_policy_url/out/wayback_errors.txt")

FROM = "20231001"
TO = "20251031"

CDX_ENDPOINT = "https://web.archive.org/cdx/search/cdx"

# 适度限速，避免被临时限制
SLEEP_SEC = 0.2

URL 扫描 + 分类 + smali 精确定位（class/method/line）

In [ ]:
URL_BYTES_RE = re.compile(
    br"https?://[A-Za-z0-9\-\._~:/\?#\[\]@!\$&'\(\)\*\+,;=%]+"
)

def normalize_url(u: str) -> str:
    return u.rstrip('"\')]>},.;')

def classify_source(rel_path: str) -> str:
    p = rel_path.replace("\\", "/")
    name = p.lower()
    if name.endswith(".dex"):
        return "dex"
    if name.endswith(".so"):
        return "lib_native"
    if p.startswith("assets/"):
        return "assets"
    if p.startswith("res/"):
        return "res"
    if name.endswith("androidmanifest.xml"):
        return "manifest"
    if name.endswith(".arsc"):
        return "resources_arsc"
    if p.startswith("smali") and name.endswith(".smali"):
        return "smali"
    if p.startswith("unknown/"):
        return "unknown"
    return "other"

def file_sha1_1mb(path: Path) -> str:
    h = hashlib.sha1()
    with path.open("rb") as f:
        h.update(f.read(1024 * 1024))
    return h.hexdigest()

def extract_urls_from_bytes(data: bytes):
    for m in URL_BYTES_RE.finditer(data):
        try:
            s = m.group(0).decode("utf-8", errors="ignore")
        except Exception:
            s = m.group(0).decode("latin1", errors="ignore")
        s = normalize_url(s)
        if s:
            yield s, m.start()

def scan_all_files(root: Path, max_mb: int = 50):
    max_bytes = max_mb * 1024 * 1024
    recs = []
    for p in root.rglob("*"):
        if not p.is_file():
            continue
        try:
            size = p.stat().st_size
            if size > max_bytes:
                continue
            data = p.read_bytes()
        except Exception:
            continue

        hits = list(extract_urls_from_bytes(data))
        if not hits:
            continue

        rel = str(p.relative_to(root)).replace("\\", "/")
        src = classify_source(rel)
        sha = file_sha1_1mb(p)

        for url, off in hits:
            recs.append({
                "url": url,
                "file": rel,
                "source": src,
                "offset": off,
                "size_bytes": size,
                "file_sha1_1mb": sha,
            })
    # dedup by (url, file, offset)
    seen = set()
    out = []
    for r in recs:
        k = (r["url"], r["file"], r["offset"])
        if k in seen:
            continue
        seen.add(k)
        out.append(r)
    return out

# ---- smali precise mapping ----
SMALI_CLASS_RE = re.compile(r"^\.class\b.*\s(L.+;)\s*$")
SMALI_METHOD_RE = re.compile(r"^\.method\b(.*)$")
SMALI_END_METHOD_RE = re.compile(r"^\.end method\b")
SMALI_CONST_STRING_RE = re.compile(r'^\s*const-string(?:/jumbo)?\s+[^,]+,\s+"(.*)"\s*$')

def map_urls_in_smali(root: Path, target_urls: set):
    usage = defaultdict(list)
    smali_dirs = [p for p in root.iterdir() if p.is_dir() and p.name.startswith("smali")]
    for sd in smali_dirs:
        for smali_file in sd.rglob("*.smali"):
            try:
                lines = smali_file.read_text(encoding="utf-8", errors="ignore").splitlines()
            except Exception:
                continue

            cur_class = None
            cur_method = None
            in_method = False

            for idx, line in enumerate(lines, start=1):
                m = SMALI_CLASS_RE.match(line)
                if m:
                    cur_class = m.group(1)

                m = SMALI_METHOD_RE.match(line)
                if m:
                    cur_method = m.group(1).strip()
                    in_method = True

                if SMALI_END_METHOD_RE.match(line):
                    in_method = False
                    cur_method = None

                m = SMALI_CONST_STRING_RE.match(line)
                if m:
                    s = m.group(1)
                    if "http://" in s or "https://" in s:
                        for url, _ in extract_urls_from_bytes(s.encode("utf-8", errors="ignore")):
                            if url in target_urls:
                                usage[url].append({
                                    "engine": "smali",
                                    "smali_file": str(smali_file.relative_to(root)).replace("\\", "/"),
                                    "class": cur_class,
                                    "method": cur_method if in_method else None,
                                    "line": idx,
                                    "snippet": line.strip(),
                                })
    return usage

# ---- run ----
records = scan_all_files(ROOT, max_mb=50)
target_urls = set(r["url"] for r in records)
usage_map = map_urls_in_smali(ROOT, target_urls)

print("Total hits:", len(records))
print("Unique URLs:", len(target_urls))
print("URLs with smali usage:", sum(1 for u in target_urls if u in usage_map))


按来源分类 + 输出 JSON / CSV

In [ ]:
by_source = Counter(r["source"] for r in records)
print("By source:", by_source)

urls_sorted = sorted(target_urls)

payload = {
    "generated_at": datetime.now().isoformat(timespec="seconds"),
    "summary": {
        "total_hits": len(records),
        "unique_urls": len(urls_sorted),
        "by_source": dict(by_source),
    },
    "urls": [
        {
            "url": u,
            "hits": [r for r in records if r["url"] == u],
            "usage": usage_map.get(u, []),
        }
        for u in urls_sorted
    ],
}

JSON_PATH.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
print("Wrote JSON:", JSON_PATH)

# CSV (flat)
fields = [
    "url","source","file","offset","size_bytes","file_sha1_1mb",
    "usage_engine","usage_smali_file","usage_class","usage_method","usage_line"
]
with CSV_PATH.open("w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=fields)
    w.writeheader()
    for r in records:
        usages = usage_map.get(r["url"], [])
        if not usages:
            w.writerow({**r,
                        "usage_engine":"",
                        "usage_smali_file":"",
                        "usage_class":"",
                        "usage_method":"",
                        "usage_line":""})
        else:
            for u in usages:
                w.writerow({**r,
                            "usage_engine": u.get("engine",""),
                            "usage_smali_file": u.get("smali_file",""),
                            "usage_class": u.get("class",""),
                            "usage_method": u.get("method",""),
                            "usage_line": u.get("line","")})

print("Wrote CSV :", CSV_PATH)


快速筛“疑似隐私政策 URL”

In [ ]:
privacy_keywords = ("privacy", "privacy-policy", "privacypolicy", "gdpr", "data-protection")

candidates = []
for u in urls_sorted:
    lu = u.lower()
    if any(k in lu for k in privacy_keywords):
        candidates.append(u)

print("Privacy-like candidates:", len(candidates))
for u in candidates[:50]:
    print("-", u)


In [ ]:
import re
from pathlib import Path
import json
from collections import defaultdict

# ===== 你只需要改这里 =====
# ROOT = Path(r"apk")  # 4个已解包目录所在的根目录
ROOT = Path(r"1122apk/test")
FOUND_TXT = Path("1122apk/1122apk_privacy_policy_url/out/found_privacy_policy_url.txt")
NOT_FOUND_TXT = Path("1122apk/1122apk_privacy_policy_url/out/not_found_Privacy_policy_url.txt")

KEYWORD_RE = re.compile(r"privacy", re.IGNORECASE)
URL_RE = re.compile(r"https?://[^\s\"\'<>]+", re.IGNORECASE)
CONTEXT_LINES = 2  # 命中关键词时，前后多少行里抓URL

def read_lines(p: Path) -> list[str]:
    return p.read_text(encoding="utf-8", errors="ignore").splitlines()


def scan_file(p: Path) -> set[str]:
    """命中 privacy 的行附近提取 URL"""
    lines = read_lines(p)
    urls = set()
    for i, line in enumerate(lines):
        if KEYWORD_RE.search(line):
            start = max(0, i - CONTEXT_LINES)
            end = min(len(lines), i + CONTEXT_LINES + 1)
            block = "\n".join(lines[start:end])
            urls.update(URL_RE.findall(block))
    return urls


def scan_one_app_1(app_dir: Path) -> list[tuple[Path, list[str]]]:
    """
    返回：[(file_path, [urls...]), ...] 仅保留 urls非空
    只扫描：
      - AndroidManifest.xml
      - res/values*/strings.xml
    """
    hits = []

    manifest = app_dir / "AndroidManifest.xml"
    if manifest.exists():
        u = sorted(scan_file(manifest))
        if u:
            hits.append((manifest, u))

    res_dir = app_dir / "res"
    if res_dir.exists():
        for strings_xml in sorted(res_dir.glob("values/strings.xml")): # values*/strings.xml
            u = sorted(scan_file(strings_xml))
            if u:
                hits.append((strings_xml, u))

    return hits

def scan_one_app(app_dir: Path) -> list[tuple[Path, list[str]]]:
    """
    返回：[(file_path, [urls...]), ...] 仅保留 urls非空
    只扫描：
      - classesx.dex
    """
    hits = []

    classesx_dir = app_dir / "AndroidManifest.xml"
    if manifest.exists():
        u = sorted(scan_file(manifest))
        if u:
            hits.append((manifest, u))

    res_dir = app_dir / "res"
    if res_dir.exists():
        for strings_xml in sorted(res_dir.glob("values/strings.xml")): # values*/strings.xml
            u = sorted(scan_file(strings_xml))
            if u:
                hits.append((strings_xml, u))

    return hits
def main():
    found_lines = []
    not_found = []

    app_dirs = [p for p in ROOT.iterdir() if p.is_dir()]
    if not app_dirs:
        raise SystemExit(f"ROOT 目录下没有子目录：{ROOT}")

    for app_dir in sorted(app_dirs):
        hits = scan_one_app(app_dir)

        if not hits:
            not_found.append(app_dir.name)
            continue

        # 你要的结构：[apkname，[[文件路径，urls], [文件路径，urls]]]
        # 为了更稳妥可解析，这里用 JSON 输出（txt 每行一条 JSON）
        payload = [
            app_dir.name,
            [[str(fp), urls] for fp, urls in hits]
        ]
        found_lines.append(json.dumps(payload, ensure_ascii=False))

    FOUND_TXT.write_text("\n".join(found_lines), encoding="utf-8")
    NOT_FOUND_TXT.write_text("\n".join(not_found), encoding="utf-8")

    print("Done.")
    print(f"found -> {FOUND_TXT.resolve()}")
    print(f"found: {len(found_lines)} lines -> {FOUND_TXT.resolve()}")
    print(f"not_found -> {NOT_FOUND_TXT.resolve()}")
    print(f"not_found: {len(not_found)} apps -> {NOT_FOUND_TXT.resolve()}")
if __name__ == "__main__":
    main()